<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch10_ex13_ex14_ex15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10: Exercise 12
Find the gradient vector of f(x,y)=sin(x^2 y) at (1.2,3.4)

In [1]:
import torch


In [2]:
x=torch.tensor(1.2, requires_grad=True)
y=torch.tensor(3.4, requires_grad=True)
f=torch.sin(x**2*y)
f.backward()
x.grad.item(), y.grad.item()


(1.489864706993103, 0.26291730999946594)

# Exercise 14
Create a custom `Dense` module that replicates the functionality of an `nn.Linear` module followed by an `nn.ReLu` module. Try implementing it first using the modules just mentioned, then reimplement it using `nn.Parameter` and the the `relu()` function.

In [3]:
import torch.nn as nn

# Custom modules: p.340

class Dense(nn.Module):
  def __init__(self, in_features, out_features):
    super().__init__()
    self.stack=nn.Sequential(
        nn.Linear(in_features, out_features),
        nn.ReLU()
    )
  def forward(self, X):
    return self.stack(X)



In [4]:
import torch.nn.functional as F


class Dense2(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()
        # ------------------------------------------------------------------
        # Create the trainable WEIGHT matrix.
        #
        # Shape: (out_features, in_features)
        # One row per neuron (output), one column per input feature (transpose of the matrix from Chapter 9)
        #
        #        input features
        #      ┌────────────────────┐
        # out0 │ w00 w01 ... w09    │
        # out1 │ w10 w11 ... w19    │
        # out2 │ w20 w21 ... w29    │
        #      └────────────────────┘
        #
        # nn.Parameter tells PyTorch:
        #
        #   "This tensor should be learned during training."
        #
        # Without nn.Parameter, this would just be an ordinary tensor and
        # the optimizer would completely ignore it.
        # ------------------------------------------------------------------
        self.weight = nn.Parameter(
            torch.randn(out_features, in_features)
        )

        # ------------------------------------------------------------------
        # Create one bias per output neuron.
        #
        # Shape: (out_features,)
        #
        # Example:
        #
        # bias = [b0, b1, b2]
        #
        # During the forward pass, this vector is automatically added to every
        # sample in the batch thanks to broadcasting.
        #
        # Biases are often initialized to zero.
        # ------------------------------------------------------------------
        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

    def forward(self, X):
        # ------------------------------------------------------------------
        # Compute the linear transformation.
        # In chapter 9: we say that the output of a layer is
        # phi(XW+b) and here the weight matrix is transposed so we should transpose it.
        #
        #
        # But we can also see why: one row of X in an instance, so, its
        # features should be multiplied with the different weights
        # of the SAME neuron, which here lie on the same row:
        # there fore W needs to be transposed.
        #
        # Finally, the bias vector of shape is added to every row
        # automatically by broadcasting.
        # ------------------------------------------------------------------
        Z = X @ self.weight.T + self.bias

        # ------------------------------------------------------------------
        # Apply the ReLU activation.
        #
        # ReLU(x) = max(0, x)
        #
        # Every negative value becomes zero while positive values are left
        # unchanged.
        #
        # We use the functional implementation instead of nn.ReLU().
        # ------------------------------------------------------------------
        return F.relu(Z)


# Exercise 15
Build and train a classification MLP on the CoverType Dataset.

a. Load the dataset and create a custom PyTorch dataset for this data
b. Create data loaders for training, validation and testing.

In [5]:

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

covtype = fetch_covtype()
X = covtype.data
y = covtype.target

StandardScaler=StandardScaler()

X=StandardScaler.fit_transform(X)
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val=train_test_split(X_test, y_test, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)

(464809, 54) (58102, 54) (58101, 54)


In [6]:
y_train[:10]

array([1, 1, 2, 2, 1, 1, 2, 2, 1, 2], dtype=int32)

In [7]:
# convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
X_val_tensor = torch.FloatTensor(X_val)
X_test_tensor = torch.FloatTensor(X_test)

# Targets must be start from 0 and need to be LongTensors for classification
y_train_tensor = torch.tensor(y_train - 1, dtype=torch.long)
y_val_tensor = torch.tensor(y_val - 1, dtype=torch.long)
y_test_tensor = torch.tensor(y_test - 1, dtype=torch.long)

In [8]:
# in order to implement mini-batch we need a DataLoader, and DataLoader requires the
# tensors to be wrapped in a dataset object

from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(X_train_tensor, y_train_tensor)
val_data = TensorDataset(X_val_tensor, y_val_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)


sample0, target0 = train_data[0]
print(sample0.shape, target0.shape)
# torch.Size([54]) (1 dimensional, 54 entries)
# torch.Size([]) (0 dimensional, one entry)


torch.Size([54]) torch.Size([])


In [9]:
# Check the absolute min and max of your targets right now
print("Min target value:", y_train_tensor.min())
print("Max target value:", y_train_tensor.max())

Min target value: tensor(0)
Max target value: tensor(6)


In [10]:
# let's create the DataLoaders

batch_size = 256


train_loader = DataLoader(train_data, batch_size = batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size = batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size = batch_size, shuffle = False)


c. Build a custom MLP module to tackle the classification task. You can optionally use the the custom Dense module from the previous exercise

In [11]:
n_inputs = 54
n_classes = 7

class CoverClassifier(nn.Module):
  def __init__(self, n_inputs, n_hidden1, n_hidden2, n_hidden3, n_classes):
    super().__init__()
    self.mlp=nn.Sequential(
        nn.Linear(n_inputs, n_hidden1), #e.g 54-200
        nn.ReLU(),
        nn.Linear(n_hidden1, n_hidden2), #e.g 200-100
        nn.ReLU(),
        nn.Linear(n_hidden2, n_hidden3), #e.g 100-50
        nn.ReLU(),
        nn.Linear(n_hidden3, n_classes) #e.g 50-7
    )
  def forward(self, X):
    return self.mlp(X)

# Note that the output layer must not use any activation function
# since we will use the nn.CrossEntropyLoss.

torch.manual_seed(42)
model = CoverClassifier(n_inputs=n_inputs,
                        n_hidden1=200,
                        n_hidden2=100,
                        n_hidden3=50,
                        n_classes=n_classes)




In [12]:
pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.8 MB/s eta 0:00:00


In [13]:
# This evaluation function checks how our model performs on a given dataset
# From notes of Chapter 10
import torchmetrics

def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch) #update at each iteration
  return metric.compute()

In [16]:
# training function (from the notes of Chapter 10)

device = 'cuda' if torch.cuda.is_available() else 'cpu'


def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
  # Initialize lists to store history for plotting or tracking later
  train_losses = []
  train_metrics = []
  eval_metrics = []

  for epoch in range(n_epochs):

    # We MUST force the model back into training mode at the start of every epoch.
    model.train()

    # We wipe the metric memory clean before the new epoch starts.
    metric.reset()

    total_loss = 0
    for X_batch, y_batch in train_loader:
      # Send data to GPU/CPU memory
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      # Forward pass: get model predictions
      y_pred = model(X_batch)

      # Calculate how wrong the model is (Loss value)
      loss = criterion(y_pred, y_batch)

      # We use '.item()' to extract the raw Python number from the loss tensor.
      total_loss += loss.item()

      # Backpropagation: Calculate gradients and update model parameters
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      # We feed the current batch's predictions and targets into the tracker.

      metric.update(y_pred, y_batch)

    # Calculate average loss for this training epoch
    mean_loss = total_loss / len(train_loader)
    train_losses.append(mean_loss)

    # We call '.compute()' to calculate the single final score for the epoch,
    train_metrics.append(metric.compute().item())

    # Run the evaluation function on the separate validation data loader.
    eval_metrics.append(evaluate_tm(model, valid_loader, metric).item())

    # Print progress report for the current epoch
    print(f'Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, '
          f'Train metric: {train_metrics[-1]:.4f}, '
          f'Valid metric: {eval_metrics[-1]:.4f}')
  return train_losses, train_metrics, eval_metrics


In [15]:
# Move model to device
model = model.to(device)

# Hyperparameters
learning_rate = 0.02
n_epochs = 20

# Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Loss function (criterion)
criterion = nn.CrossEntropyLoss().to(device)

# Metric (evaluation metric)
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

# Run training loop
train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)



Epoch 1/20, Loss: 0.8678, Train metric: 0.6516, Valid metric: 0.7323
Epoch 2/20, Loss: 0.6359, Train metric: 0.7383, Valid metric: 0.7458
Epoch 3/20, Loss: 0.5964, Train metric: 0.7502, Valid metric: 0.7560
Epoch 4/20, Loss: 0.5691, Train metric: 0.7604, Valid metric: 0.7665
Epoch 5/20, Loss: 0.5472, Train metric: 0.7684, Valid metric: 0.7739
Epoch 6/20, Loss: 0.5279, Train metric: 0.7756, Valid metric: 0.7780
Epoch 7/20, Loss: 0.5098, Train metric: 0.7833, Valid metric: 0.7915
Epoch 8/20, Loss: 0.4929, Train metric: 0.7916, Valid metric: 0.7937
Epoch 9/20, Loss: 0.4794, Train metric: 0.7972, Valid metric: 0.7728
Epoch 10/20, Loss: 0.4664, Train metric: 0.8028, Valid metric: 0.8099
Epoch 11/20, Loss: 0.4556, Train metric: 0.8077, Valid metric: 0.8039
Epoch 12/20, Loss: 0.4450, Train metric: 0.8123, Valid metric: 0.7828
Epoch 13/20, Loss: 0.4359, Train metric: 0.8164, Valid metric: 0.8177
Epoch 14/20, Loss: 0.4278, Train metric: 0.8198, Valid metric: 0.8211
Epoch 15/20, Loss: 0.4195, Tr

([0.8678233301193179,
  0.6359362100714628,
  0.5964151237668708,
  0.5691250680170395,
  0.5472135795895748,
  0.5278553519283097,
  0.5097513785708844,
  0.4928587172590688,
  0.47935873778774873,
  0.46641332624964255,
  0.4556087159637838,
  0.4449904228884743,
  0.43593603415242377,
  0.42780895685440645,
  0.41945820157354624,
  0.41168649793327644,
  0.4051143625824987,
  0.3981416310113957,
  0.3919616348319379,
  0.38416242881755996],
 [0.6516289710998535,
  0.7382537722587585,
  0.750200629234314,
  0.760411262512207,
  0.7683736681938171,
  0.7756088972091675,
  0.7833153009414673,
  0.7915810346603394,
  0.7971790432929993,
  0.8028437495231628,
  0.8077102899551392,
  0.8123229146003723,
  0.8163568377494812,
  0.8198356628417969,
  0.8240094184875488,
  0.8274280428886414,
  0.8309606909751892,
  0.8343900442123413,
  0.8368878364562988,
  0.8407732844352722],
 [tensor(0.7323, device='cuda:0'),
  tensor(0.7458, device='cuda:0'),
  tensor(0.7560, device='cuda:0'),
  tensor

In [17]:
torch.manual_seed(42)

# re-initialize the model
model = CoverClassifier(n_inputs=n_inputs,
                        n_hidden1=200,
                        n_hidden2=100,
                        n_hidden3=50,
                        n_classes=n_classes)
# Move model to device
model = model.to(device)

# Hyperparameter epochs
n_epochs = 15

# We run several loops with smaller and smaller learning rates
for learning_rate in [0.16, 0.08, 0.04, 0.02, 0.01, 0.005]:
  # Optimizer
  optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

  # Loss function (criterion)
  criterion = nn.CrossEntropyLoss().to(device)

  # Metric (evaluation metric)
  accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

  # Run training loop
  print(f"--------- Learning Rate: {learning_rate}")
  # catch the return of the train function so it does not print
  _ = train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)

--------- Learning Rate: 0.16
Epoch 1/15, Loss: 0.6324, Train metric: 0.7339, Valid metric: 0.7698
Epoch 2/15, Loss: 0.5113, Train metric: 0.7789, Valid metric: 0.7866
Epoch 3/15, Loss: 0.4567, Train metric: 0.8053, Valid metric: 0.8147
Epoch 4/15, Loss: 0.4135, Train metric: 0.8250, Valid metric: 0.8356
Epoch 5/15, Loss: 0.3806, Train metric: 0.8400, Valid metric: 0.8453
Epoch 6/15, Loss: 0.3540, Train metric: 0.8527, Valid metric: 0.8457
Epoch 7/15, Loss: 0.3320, Train metric: 0.8619, Valid metric: 0.8701
Epoch 8/15, Loss: 0.3139, Train metric: 0.8702, Valid metric: 0.8745
Epoch 9/15, Loss: 0.2999, Train metric: 0.8764, Valid metric: 0.8801
Epoch 10/15, Loss: 0.2860, Train metric: 0.8824, Valid metric: 0.8784
Epoch 11/15, Loss: 0.2761, Train metric: 0.8864, Valid metric: 0.8898
Epoch 12/15, Loss: 0.2659, Train metric: 0.8910, Valid metric: 0.8829
Epoch 13/15, Loss: 0.2574, Train metric: 0.8949, Valid metric: 0.8835
Epoch 14/15, Loss: 0.2497, Train metric: 0.8984, Valid metric: 0.9001

In [19]:
# Let's test the model on the test set
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)
evaluate_tm(model, test_loader, accuracy)

tensor(0.9499, device='cuda:0')